# 301 · Row vs columnar experiment

Companion to [Row vs columnar at system scale](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/301/row-vs-columnar/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/301/row_vs_columnar.ipynb)

**Goal:** measure toy I/O when the workload is “whole event” vs “one column over many rows.”

> **Honesty banner:** notebook experiments implement the article **Experiments** blocks. They do not replace suite Results. Timings/sizes are illustrative.



In [ ]:
import json
import timeit
from statistics import median

N = 2000
WIDTH = 30  # wide table
rows = [
    {f"c{j}": (i * 31 + j) % 997 for j in range(WIDTH)} | {"event_id": i, "sku": f"S{i%50}"}
    for i in range(N)
]

# Row-oriented encoding: JSONL (stand-in for per-event messages)
jsonl = "\n".join(json.dumps(r, separators=(",", ":")) for r in rows).encode()

# Column-oriented: dict of arrays (stand-in for Arrow/Parquet column pages)
cols = {k: [r[k] for r in rows] for k in rows[0]}

print("JSONL nbytes:", len(jsonl))
print("column 'c0' alone nbytes (json list):", len(json.dumps(cols["c0"]).encode()))



In [ ]:
def sum_c0_via_jsonl():
    total = 0
    for line in jsonl.splitlines():
        total += json.loads(line)["c0"]
    return total


def sum_c0_via_cols():
    return sum(cols["c0"])


def fetch_one_event_row(i=123):
    # scan JSONL for one id (bad lake pattern, realistic if you dump RPC to files)
    for line in jsonl.splitlines():
        o = json.loads(line)
        if o["event_id"] == i:
            return o
    return None


def fetch_one_event_cols(i=123):
    # still need a row index — point lookup wants a row store / key-value
    idx = cols["event_id"].index(i)
    return {k: cols[k][idx] for k in cols}


t_scan_row = median(timeit.repeat(sum_c0_via_jsonl, number=1, repeat=3))
t_scan_col = median(timeit.repeat(sum_c0_via_cols, number=50, repeat=3))
t_point_row = median(timeit.repeat(fetch_one_event_row, number=5, repeat=3))
t_point_col = median(timeit.repeat(fetch_one_event_cols, number=5, repeat=3))

assert sum_c0_via_jsonl() == sum_c0_via_cols()
print(f"scan sum(c0) via JSONL lines: {t_scan_row*1e3:.2f} ms")
print(f"scan sum(c0) via column:      {t_scan_col*1e3:.2f} ms (50× loop in timer)")
print(f"point get event via JSONL scan: {t_point_row*1e3:.2f} ms")
print(f"point get via column index:     {t_point_col*1e3:.2f} ms")
print("Architecture: events/RPC = row messages; lake scans = columnar files—not one format for both.")



## Decision table (article)

| Workload | Default |
|----------|---------|
| RPC / per-event stream | Row |
| Nightly lake on object storage | Columnar |
| Cache get-by-id | Row |
| Feature training wide tables | Columnar |

